In [1]:
import os, re
import torch;

In [ ]:
import os, re
import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==5.2.0
!pip install --no-deps trl==0.22.2
!pip install tensorboard
!pip install openpyxl

In [4]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())

Torch version: 2.8.0+cu128
CUDA available: True
Torch CUDA version: 12.8
GPU count: 8


/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [5]:
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_DISABLE"] = "1"

In [6]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "../Models/VLM-unsloth-8b",
    load_in_4bit = False,
    load_in_16bit = True,
    dtype = torch.bfloat16,
    use_gradient_checkpointing = "unsloth", 
)

/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/usr/local/lib/python3.10/dist-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
[fla.utils|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Qwen3_Vl patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.109 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 750/750 [1:03:26<00:00,  5.08s/it, Materializing param=model.visual.pos_embed.weight]                                 


In [7]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == 2r
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = True,  # Better stabilized Training
    loftq_config = None, # And LoftQ
)

In [ ]:
import pandas as pd
import ast
from pathlib import Path



val_csv_path = r'valid.csv'
train_csv_path = r'train.csv'

# Load the CSV files
train_df = pd.read_csv(train_csv_path)
val_df = pd.read_csv(val_csv_path)

# --- 4. Print Summary ---
print("✅ Datasets loaded and processed successfully.\n")
print(f"Training samples: {len(train_df):,}")
print("Training set distribution:\n", train_df['image_path'].value_counts())
print("-" * 30)
print(f"Validation samples: {len(val_df):,}")
print("Validation set distribution:\n", train_df['label'].value_counts())
print("-" * 30)

# --- 5. Preview a few rows ---
print(train_df.head())


✅ Datasets loaded and processed successfully.

Training samples: 67,514
Training set distribution:
 image_path
dataset/train/XR_WRIST/patient09082/study1_negative/augmentation/image3_aug.png                     1
dataset/train/XR_ELBOW/patient00011/study1_negative/image3.png                                      1
dataset/train/XR_ELBOW/patient00011/study1_negative/augmentation/image1_aug.png                     1
dataset/train/XR_ELBOW/patient00011/study1_negative/augmentation/image3_aug.png                     1
dataset/train/XR_ELBOW/patient00011/study1_negative/augmentation/augmentation/image1_aug_aug.png    1
                                                                                                   ..
dataset/train/XR_ELBOW/patient00034/study1_positive/augmentation/image1_aug.png                     1
dataset/train/XR_ELBOW/patient00034/study1_positive/augmentation/image2_aug.png                     1
dataset/train/XR_ELBOW/patient00034/study1_positive/augmentation/image3_a

In [10]:
from datasets import Dataset, DatasetDict, Features, Value, Sequence, Image
from PIL import Image as PILImage
import io
from torchvision import transforms

# 1️⃣ Define dataset features explicitly
features = Features({
    "image_path": Value("string"),
    "label": Value("string"),
    "modality": Value("string"),
})

# 2️⃣ Convert pandas DataFrames to HF Datasets
train_dataset = Dataset.from_pandas(train_df, features=features)
val_dataset = Dataset.from_pandas(val_df, features=features)


# 4️⃣ Combine into DatasetDict
data = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

# 5️⃣ Verify
print(train_dataset[0]["image_path"])
print(type(train_dataset[0]["image_path"])) 
print(val_dataset[0]["label"])
print(type(val_dataset[0]["label"]))  


dataset/train/XR_ELBOW/patient00011/study1_negative/image3.png
<class 'str'>
positive
<class 'str'>


In [11]:
# ----------------------------
# Instruction for Pneumothorax
# ----------------------------
instruction = """You are a medical imaging assistant specialized in musculoskeletal X-ray interpretation.

You will be given the X-ray images.

Task:
Classify the image as either positive or negative.

Definitions:
- Positive: The image shows abnormality.
- Negative: The image shows no abnormality.

Rules:
- Do not use information outside the given image.
- Do not guess or speculate.
- Do not explain your reasoning.
- Do not output anything except the final label.

Output:
Respond with exactly one word, all lowercase.

Allowed outputs:
positive
negative
"""


# ----------------------------
# Convert dataset row into conversation format
# ----------------------------
def convert_to_conversation(sample):
    """
    sample: dict with keys:
    - image_path
    - modality
    - label
    """
    
    user_text = (
        f"{instruction}\n"
        f"Image Modality: {sample['modality']}"
    )

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_text},
                {"type": "image", "image": sample["image_path"]}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": sample["label"]}
            ]
        }
    ]

    return {"messages": conversation}



In [12]:
converted_dataset_test = [convert_to_conversation(sample) for sample in train_dataset]
converted_dataset_val = [convert_to_conversation(sample) for sample in val_dataset]

In [13]:
converted_dataset_test[3]
converted_dataset_val[3]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'You are a medical imaging assistant specialized in musculoskeletal X-ray interpretation.\n\nYou will be given the X-ray images.\n\nTask:\nClassify the image as either positive or negative.\n\nDefinitions:\n- Positive: The image shows abnormality.\n- Negative: The image shows no abnormality.\n\nRules:\n- Do not use information outside the given image.\n- Do not guess or speculate.\n- Do not explain your reasoning.\n- Do not output anything except the final label.\n\nOutput:\nRespond with exactly one word, all lowercase.\n\nAllowed outputs:\npositive\nnegative\n\nImage Modality: XR_ELBOW'},
    {'type': 'image',
     'image': 'dataset/valid/XR_ELBOW/patient11186/study1_positive/image4.png'}]},
  {'role': 'assistant', 'content': [{'type': 'text', 'text': 'positive'}]}]}

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset_test,
    eval_dataset= converted_dataset_val,

    args = SFTConfig(
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.05,
        num_train_epochs = 6, 
        learning_rate = 1e-4,
        logging_steps = 10,
        optim = "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs-16-32-6-1e4-8b-16bit",
        report_to = "none",     # For Weights and Biases
        eval_strategy="steps",      # Run evaluation on the validation set at the end of each epoch.
        eval_steps=200,
        save_strategy= "epoch",
        bf16=True,  # if supported

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
        torch_compile=False,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 67,514 | Num Epochs = 6 | Total steps = 12,660
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 51,346,944 of 8,818,470,640 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
200,0.018560,0.020718
400,0.018101,0.020074
600,0.018163,0.019889
800,0.018326,0.019244
1000,0.018204,0.019296
1200,0.018423,0.019361
1400,0.017786,0.019386
1600,0.018018,0.019143
1800,0.018138,0.019160
2000,0.017945,0.019674


Unsloth: Not an error, but Qwen3VLForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

